# Task 3: Model Development

## Objective
This notebook covers:
1. Data preparation (encoding, scaling, splitting)
2. Implementation of 4 model families:
   - Linear Model: Logistic Regression
   - Tree-based Model: Random Forest
   - Boosting Model: XGBoost and LightGBM
   - Advanced Model: Neural Network (PyTorch)
3. Hyperparameter tuning with cross-validation
4. MLflow experiment tracking
5. Model saving and comparison

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

# Boosting
import xgboost as xgb
import lightgbm as lgb

# Imbalanced learning
from imblearn.over_sampling import SMOTE

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.pytorch

# Utils
import joblib
import os
from datetime import datetime

# Set display and plot options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3.1 Load Processed Data

In [ ]:
# Load processed data from Task 2
df = pd.read_csv('../data/interim/bank_processed.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nTarget distribution:")
print(df['y'].value_counts())
print(f"\nClass balance: {df['y'].value_counts()['no'] / df['y'].value_counts()['yes']:.2f}:1")

## 3.2 Data Preparation

### Strategy
1. **Separate features and target**
2. **Identify feature types**: numeric, categorical
3. **Create preprocessing pipeline**: StandardScaler for numeric, OneHotEncoder for categorical
4. **Train-test split**: 80-20 with stratification
5. **Apply SMOTE**: Only on training set to avoid data leakage

In [ ]:
# Separate features and target
# Exclude: 'y' (original target), 'y_encoded' (we'll encode fresh), 'data_source' (not predictive)
exclude_cols = ['y', 'y_encoded', 'data_source']
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols].copy()
y = df['y'].copy()

# Encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y_encoded.shape}")
print(f"\nTarget encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")
print(f"\nFeature columns ({len(feature_cols)}):")
print(feature_cols)

In [ ]:
# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")

In [ ]:
# Create preprocessing pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Numeric: StandardScaler
# Categorical: OneHotEncoder with handle_unknown='ignore'
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

print("Preprocessing pipeline created!")
print(f"  - Numeric features: StandardScaler")
print(f"  - Categorical features: OneHotEncoder (drop first, handle unknown)")

In [ ]:
# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")

In [ ]:
# Fit preprocessor on training data and transform both sets
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

print(f"Training set after preprocessing: {X_train_scaled.shape}")
print(f"Test set after preprocessing: {X_test_scaled.shape}")

# Get feature names after one-hot encoding
try:
    feature_names = (numeric_features + 
                    preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist())
    print(f"\nTotal features after encoding: {len(feature_names)}")
except:
    feature_names = None
    print("\nCould not extract feature names (will use indices)")

### 3.2.1 Apply SMOTE for Balanced Training

We apply SMOTE only to the training set to create a balanced dataset for training.

In [ ]:
# Apply SMOTE to balance training data
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:")
print(f"  Training set shape: {X_train_scaled.shape}")
print(f"  Class distribution: {np.bincount(y_train)}")

print("\nAfter SMOTE:")
print(f"  Training set shape: {X_train_balanced.shape}")
print(f"  Class distribution: {np.bincount(y_train_balanced)}")
print(f"  ✓ Classes are now balanced!")

# Visualize before and after
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before SMOTE
axes[0].bar(['Class 0 (no)', 'Class 1 (yes)'], np.bincount(y_train), color=['#ff7f0e', '#2ca02c'])
axes[0].set_title('Before SMOTE', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12)
for i, v in enumerate(np.bincount(y_train)):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# After SMOTE
axes[1].bar(['Class 0 (no)', 'Class 1 (yes)'], np.bincount(y_train_balanced), color=['#ff7f0e', '#2ca02c'])
axes[1].set_title('After SMOTE', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count', fontsize=12)
for i, v in enumerate(np.bincount(y_train_balanced)):
    axes[1].text(i, v + 1000, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/03_smote_balance.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.3 Setup MLflow Experiment Tracking

In [ ]:
# Setup MLflow
os.makedirs('../experiments/mlruns', exist_ok=True)
mlflow.set_tracking_uri('file:../experiments/mlruns')
mlflow.set_experiment('bank_marketing_classification')

print("MLflow experiment tracking configured!")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('bank_marketing_classification')}")

In [ ]:
# Helper function to log metrics
def evaluate_and_log_model(model, model_name, X_train, y_train, X_test, y_test, log_to_mlflow=True):
    """
    Evaluate model and log metrics to MLflow.
    """
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Probabilities (if available)
    try:
        y_train_proba = model.predict_proba(X_train)[:, 1]
        y_test_proba = model.predict_proba(X_test)[:, 1]
    except:
        y_train_proba = y_train_pred
        y_test_proba = y_test_pred
    
    # Calculate metrics
    metrics = {
        'train_accuracy': accuracy_score(y_train, y_train_pred),
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'train_precision': precision_score(y_train, y_train_pred, zero_division=0),
        'test_precision': precision_score(y_test, y_test_pred, zero_division=0),
        'train_recall': recall_score(y_train, y_train_pred, zero_division=0),
        'test_recall': recall_score(y_test, y_test_pred, zero_division=0),
        'train_f1': f1_score(y_train, y_train_pred, zero_division=0),
        'test_f1': f1_score(y_test, y_test_pred, zero_division=0),
        'train_roc_auc': roc_auc_score(y_train, y_train_proba),
        'test_roc_auc': roc_auc_score(y_test, y_test_proba)
    }
    
    # Log to MLflow
    if log_to_mlflow:
        with mlflow.start_run(run_name=model_name):
            # Log parameters
            if hasattr(model, 'get_params'):
                params = model.get_params()
                for param, value in params.items():
                    try:
                        mlflow.log_param(param, value)
                    except:
                        pass
            
            # Log metrics
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(metric_name, metric_value)
            
            # Log model
            try:
                mlflow.sklearn.log_model(model, model_name)
            except:
                pass
    
    return metrics, y_test_pred, y_test_proba

print("Helper function created!")

## 3.4 Model 1: Logistic Regression (Linear Model)

### Justification
- **Interpretable**: Coefficients show feature importance
- **Fast**: Quick training and prediction
- **Baseline**: Good baseline for comparison
- **Works well**: Effective for linearly separable problems

In [ ]:
print("="*80)
print("MODEL 1: LOGISTIC REGRESSION")
print("="*80)

# Hyperparameter tuning with GridSearchCV
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'max_iter': [1000]
}

lr_model = LogisticRegression(random_state=42)
grid_search_lr = GridSearchCV(
    lr_model, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)

print("Training Logistic Regression with GridSearchCV...")
grid_search_lr.fit(X_train_balanced, y_train_balanced)

print(f"\nBest parameters: {grid_search_lr.best_params_}")
print(f"Best CV ROC-AUC: {grid_search_lr.best_score_:.4f}")

# Best model
best_lr = grid_search_lr.best_estimator_

# Evaluate
lr_metrics, lr_pred, lr_proba = evaluate_and_log_model(
    best_lr, 'logistic_regression', X_train_balanced, y_train_balanced, X_test_scaled, y_test
)

print("\nLogistic Regression Performance:")
for metric, value in lr_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Save model
os.makedirs('../models', exist_ok=True)
joblib.dump(best_lr, '../models/logistic_regression.pkl')
print("\n✓ Model saved to models/logistic_regression.pkl")

## 3.5 Model 2: Random Forest (Tree-based Model)

### Justification
- **Handles non-linearity**: Captures complex relationships
- **Feature importance**: Built-in feature importance
- **Robust**: Less prone to overfitting with proper tuning
- **No scaling needed**: Works with raw features (we've scaled for consistency)

In [ ]:
print("="*80)
print("MODEL 2: RANDOM FOREST")
print("="*80)

# Hyperparameter tuning
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search_rf = GridSearchCV(
    rf_model, param_grid_rf, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)

print("Training Random Forest with GridSearchCV...")
print("Note: This may take several minutes...")
grid_search_rf.fit(X_train_balanced, y_train_balanced)

print(f"\nBest parameters: {grid_search_rf.best_params_}")
print(f"Best CV ROC-AUC: {grid_search_rf.best_score_:.4f}")

# Best model
best_rf = grid_search_rf.best_estimator_

# Evaluate
rf_metrics, rf_pred, rf_proba = evaluate_and_log_model(
    best_rf, 'random_forest', X_train_balanced, y_train_balanced, X_test_scaled, y_test
)

print("\nRandom Forest Performance:")
for metric, value in rf_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Save model
joblib.dump(best_rf, '../models/random_forest.pkl')
print("\n✓ Model saved to models/random_forest.pkl")

## 3.6 Model 3: XGBoost (Boosting Model)

### Justification
- **State-of-the-art**: Often wins competitions
- **Handles imbalance**: scale_pos_weight parameter
- **Regularization**: Built-in L1/L2 regularization
- **Fast**: Efficient implementation with GPU support

In [ ]:
print("="*80)
print("MODEL 3: XGBOOST")
print("="*80)

# Calculate scale_pos_weight for imbalance
scale_pos_weight = (y_train_balanced == 0).sum() / (y_train_balanced == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Hyperparameter tuning
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_model = xgb.XGBClassifier(
    random_state=42, 
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False
)

grid_search_xgb = GridSearchCV(
    xgb_model, param_grid_xgb, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)

print("Training XGBoost with GridSearchCV...")
print("Note: This may take several minutes...")
grid_search_xgb.fit(X_train_balanced, y_train_balanced)

print(f"\nBest parameters: {grid_search_xgb.best_params_}")
print(f"Best CV ROC-AUC: {grid_search_xgb.best_score_:.4f}")

# Best model
best_xgb = grid_search_xgb.best_estimator_

# Evaluate
xgb_metrics, xgb_pred, xgb_proba = evaluate_and_log_model(
    best_xgb, 'xgboost', X_train_balanced, y_train_balanced, X_test_scaled, y_test
)

print("\nXGBoost Performance:")
for metric, value in xgb_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Save model
best_xgb.save_model('../models/xgboost_model.json')
joblib.dump(best_xgb, '../models/xgboost.pkl')
print("\n✓ Model saved to models/xgboost.pkl and xgboost_model.json")

## 3.7 Model 4: LightGBM (Alternative Boosting Model)

### Justification
- **Faster than XGBoost**: Histogram-based algorithm
- **Memory efficient**: Lower memory usage
- **Handles categorical**: Native categorical feature support
- **Great for large datasets**: Scales well

In [ ]:
print("="*80)
print("MODEL 4: LIGHTGBM")
print("="*80)

# Hyperparameter tuning
param_grid_lgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.1, 0.3],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 1.0]
}

lgb_model = lgb.LGBMClassifier(
    random_state=42,
    is_unbalance=True,
    verbose=-1
)

grid_search_lgb = GridSearchCV(
    lgb_model, param_grid_lgb, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)

print("Training LightGBM with GridSearchCV...")
print("Note: This may take several minutes...")
grid_search_lgb.fit(X_train_balanced, y_train_balanced)

print(f"\nBest parameters: {grid_search_lgb.best_params_}")
print(f"Best CV ROC-AUC: {grid_search_lgb.best_score_:.4f}")

# Best model
best_lgb = grid_search_lgb.best_estimator_

# Evaluate
lgb_metrics, lgb_pred, lgb_proba = evaluate_and_log_model(
    best_lgb, 'lightgbm', X_train_balanced, y_train_balanced, X_test_scaled, y_test
)

print("\nLightGBM Performance:")
for metric, value in lgb_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Save model
joblib.dump(best_lgb, '../models/lightgbm.pkl')
print("\n✓ Model saved to models/lightgbm.pkl")

## 3.8 Model 5: Neural Network (Advanced Model)

### Justification
- **Deep learning**: Can learn complex non-linear patterns
- **Flexible architecture**: Can be tuned for specific problems
- **State-of-the-art**: Competitive with boosting methods
- **Regularization**: Dropout and batch normalization

In [ ]:
print("="*80)
print("MODEL 5: NEURAL NETWORK (PyTorch)")
print("="*80)

# Define Neural Network architecture
class BankMarketingNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.3):
        super(BankMarketingNN, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Prepare data for PyTorch
X_train_tensor = torch.FloatTensor(X_train_balanced)
y_train_tensor = torch.FloatTensor(y_train_balanced).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

# Initialize model
input_dim = X_train_balanced.shape[1]
model_nn = BankMarketingNN(input_dim, hidden_dims=[128, 64, 32], dropout=0.3)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model_nn.parameters(), lr=0.001)

print(f"\nNeural Network Architecture:")
print(model_nn)
print(f"\nTotal parameters: {sum(p.numel() for p in model_nn.parameters()):,}")

In [ ]:
# Training loop
print("\nTraining Neural Network...")
num_epochs = 50
train_losses = []

model_nn.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        # Forward pass
        outputs = model_nn(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("\n✓ Training completed!")

# Plot training loss
plt.figure(figsize=(10, 6))
plt.plot(train_losses, linewidth=2)
plt.title('Neural Network Training Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/03_nn_training_loss.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate Neural Network
model_nn.eval()
with torch.no_grad():
    # Training predictions
    nn_train_proba = model_nn(X_train_tensor).numpy().flatten()
    nn_train_pred = (nn_train_proba > 0.5).astype(int)
    
    # Test predictions
    nn_test_proba = model_nn(X_test_tensor).numpy().flatten()
    nn_test_pred = (nn_test_proba > 0.5).astype(int)

# Calculate metrics
nn_metrics = {
    'train_accuracy': accuracy_score(y_train_balanced, nn_train_pred),
    'test_accuracy': accuracy_score(y_test, nn_test_pred),
    'train_precision': precision_score(y_train_balanced, nn_train_pred, zero_division=0),
    'test_precision': precision_score(y_test, nn_test_pred, zero_division=0),
    'train_recall': recall_score(y_train_balanced, nn_train_pred, zero_division=0),
    'test_recall': recall_score(y_test, nn_test_pred, zero_division=0),
    'train_f1': f1_score(y_train_balanced, nn_train_pred, zero_division=0),
    'test_f1': f1_score(y_test, nn_test_pred, zero_division=0),
    'train_roc_auc': roc_auc_score(y_train_balanced, nn_train_proba),
    'test_roc_auc': roc_auc_score(y_test, nn_test_proba)
}

print("Neural Network Performance:")
for metric, value in nn_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Log to MLflow
with mlflow.start_run(run_name='neural_network'):
    mlflow.log_param('architecture', '128-64-32')
    mlflow.log_param('dropout', 0.3)
    mlflow.log_param('learning_rate', 0.001)
    mlflow.log_param('batch_size', 256)
    mlflow.log_param('epochs', num_epochs)
    
    for metric_name, metric_value in nn_metrics.items():
        mlflow.log_metric(metric_name, metric_value)
    
    # Save model
    torch.save(model_nn.state_dict(), '../models/neural_network.pth')
    mlflow.log_artifact('../models/neural_network.pth')

print("\n✓ Model saved to models/neural_network.pth")

## 3.9 Model Comparison Summary

In [ ]:
# Compile all results
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM', 'Neural Network'],
    'Test_Accuracy': [
        lr_metrics['test_accuracy'],
        rf_metrics['test_accuracy'],
        xgb_metrics['test_accuracy'],
        lgb_metrics['test_accuracy'],
        nn_metrics['test_accuracy']
    ],
    'Test_Precision': [
        lr_metrics['test_precision'],
        rf_metrics['test_precision'],
        xgb_metrics['test_precision'],
        lgb_metrics['test_precision'],
        nn_metrics['test_precision']
    ],
    'Test_Recall': [
        lr_metrics['test_recall'],
        rf_metrics['test_recall'],
        xgb_metrics['test_recall'],
        lgb_metrics['test_recall'],
        nn_metrics['test_recall']
    ],
    'Test_F1': [
        lr_metrics['test_f1'],
        rf_metrics['test_f1'],
        xgb_metrics['test_f1'],
        lgb_metrics['test_f1'],
        nn_metrics['test_f1']
    ],
    'Test_ROC_AUC': [
        lr_metrics['test_roc_auc'],
        rf_metrics['test_roc_auc'],
        xgb_metrics['test_roc_auc'],
        lgb_metrics['test_roc_auc'],
        nn_metrics['test_roc_auc']
    ]
})

print("="*80)
print("MODEL COMPARISON - TEST SET PERFORMANCE")
print("="*80)
display(results_df.round(4))

# Save results
results_df.to_csv('../reports/model_comparison.csv', index=False)
print("\n✓ Results saved to reports/model_comparison.csv")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

metrics_to_plot = ['Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1']
colors = plt.cm.Set3(range(len(results_df)))

for idx, metric in enumerate(metrics_to_plot):
    row = idx // 2
    col = idx % 2
    
    axes[row, col].bar(results_df['Model'], results_df[metric], color=colors)
    axes[row, col].set_title(metric.replace('_', ' '), fontsize=14, fontweight='bold')
    axes[row, col].set_ylabel('Score', fontsize=12)
    axes[row, col].set_ylim([0, 1])
    axes[row, col].tick_params(axis='x', rotation=45)
    
    # Add value labels
    for i, v in enumerate(results_df[metric]):
        axes[row, col].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/03_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Find best model
best_model_idx = results_df['Test_F1'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_f1 = results_df.loc[best_model_idx, 'Test_F1']

print("="*80)
print("BEST MODEL BASED ON F1-SCORE")
print("="*80)
print(f"Model: {best_model_name}")
print(f"F1-Score: {best_f1:.4f}")
print(f"\nFull metrics:")
display(results_df.loc[best_model_idx])

## 3.10 Save Preprocessing Pipeline and Data

In [ ]:
# Save preprocessing artifacts
joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump(label_encoder, '../models/label_encoder.pkl')
joblib.dump(feature_names, '../models/feature_names.pkl')

# Save train/test split for reproducibility
np.save('../data/processed/X_train_balanced.npy', X_train_balanced)
np.save('../data/processed/y_train_balanced.npy', y_train_balanced)
np.save('../data/processed/X_test_scaled.npy', X_test_scaled)
np.save('../data/processed/y_test.npy', y_test)

print("✓ Preprocessing pipeline saved!")
print("✓ Train/test data saved!")
print("\nSaved files:")
print("  - models/preprocessor.pkl")
print("  - models/label_encoder.pkl")
print("  - models/feature_names.pkl")
print("  - data/processed/X_train_balanced.npy")
print("  - data/processed/y_train_balanced.npy")
print("  - data/processed/X_test_scaled.npy")
print("  - data/processed/y_test.npy")

## 3.11 Summary

### Models Developed
1. **Logistic Regression** (Linear Model)
   - Simple, interpretable baseline
   - Hyperparameter tuning with GridSearchCV

2. **Random Forest** (Tree-based Model)
   - Ensemble of decision trees
   - Robust to overfitting with proper tuning

3. **XGBoost** (Boosting Model)
   - State-of-the-art gradient boosting
   - Excellent performance on structured data

4. **LightGBM** (Alternative Boosting Model)
   - Faster and more memory-efficient than XGBoost
   - Histogram-based algorithm

5. **Neural Network** (Advanced Model)
   - Deep learning with PyTorch
   - 3 hidden layers with batch normalization and dropout

### Key Achievements
- ✓ All models trained with SMOTE-balanced data
- ✓ Hyperparameter tuning with cross-validation
- ✓ Comprehensive metrics tracked with MLflow
- ✓ All models saved for deployment
- ✓ Comparison table generated

### Next Steps (Task 4)
1. Detailed evaluation with confusion matrices
2. ROC and Precision-Recall curves
3. Error analysis
4. Threshold tuning for optimal F1
5. Model explainability (SHAP, LIME)